In [8]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
from functools import reduce
import geopandas as gpd
from shapely.geometry import Point

In [10]:
# CSV Fold 
BASE_FOLD = Path.cwd () / "GloFASv5_Stations_Overview_260223"

# Output Dir
OUT_FOLD = Path.cwd() / "../GloFASv5_Stations_Overview_260223"

# CSVs
data1 = pd.read_csv(BASE_FOLD / "GloFASv5_catchments_comprehensive.csv")
data2 = pd.read_csv(BASE_FOLD / "GloFASv5_catchments_clim_comprehensive.csv")
data3 = pd.read_csv(BASE_FOLD / "GloFASv5_catchments_parameters_comprehensive.csv")
data4 = pd.read_csv(BASE_FOLD / "GloFASv5_catchments_glaciers_comprehensive.csv")

# station list
cal_stations = data1[["LISFLOOD_X","LISFLOOD_Y", "ID"]].values


In [3]:
dfs = [data1, data2, data3, data4]

# merge all on ID, keeping overlapping columns from the first occurrence
merged = reduce(lambda left, right: pd.merge(left, right, on="ID", suffixes=("", "_dup")), dfs)

# drop all duplicate columns (those ending with _dup)
dup_cols = [c for c in merged.columns if c.endswith("_dup")]
merged = merged.drop(columns=dup_cols)

print(f"Final shape: {merged.shape}")
# print(merged.head())

Final shape: (5379, 69)


In [ ]:
# Rename
merged = merged.rename(columns={
    "long":       "lon",
    "LISFLOOD_X": "grid_x",
    "LISFLOOD_Y": "grid_y",
    # Calibration parameters → param_ prefix
    "CalChanMan1":          "param_CalChanMan1",
    "CalChanMan3":          "param_CalChanMan3",
    "GwLoss":               "param_GwLoss",
    "GwPercValue":          "param_GwPercValue",
    "LZThreshold":          "param_LZThreshold",
    "LakeMultiplier":       "param_LakeMultiplier",
    "LowerZoneTimeConstant":"param_LowerZoneTimeConstant",
    "PowerPrefFlow":        "param_PowerPrefFlow",
    "SnowMeltCoef":         "param_SnowMeltCoef",
    "TransSub":             "param_TransSub",
    "UpperZoneTimeConstant":"param_UpperZoneTimeConstant",
    "b_Xinanjiang":         "param_b_Xinanjiang",
})

# Drop redundant Basin column
merged = merged.drop(columns=["Basin"])
merged.loc[merged['lat'].abs() > 90, 'lat'] = merged.loc[merged['lat'].abs() > 90, 'grid_y']
merged.loc[merged['lon'].abs() > 180, 'lon'] = merged.loc[merged['lon'].abs() > 180, 'grid_y']

# Store as CSV
merged.to_csv("glofas5_hydrobot.csv", index=False)
merged.to_csv(OUT_FOLD / "glofas5_hydrobot.csv", index=False)
print(merged.columns.tolist())
print(merged.shape)
print("Saved!")

['ID', 'name', 'basin', 'river', 'provider', 'iso', 'DrainageArea_prov', 'DrainageArea_LDD', 'lat', 'lon', 'grid_x', 'grid_y', 'SourceGlo', 'GlofasV5', 'Obs_start', 'Obs_end', 'Split_date_CALstart', 'KGEmod', 'JSD', 'Function', 'Region', 'elv_mean', 'elv_median', 'laii_mean', 'laii_median', 'laif_mean', 'laif_median', 'gradient_mean', 'gradient_median', 'lusemask_mean', 'lusemask_median', 'soildepth1_f_mean', 'soildepth1_f_median', 'soildepth1_o_mean', 'soildepth1_o_median', 'fracforest_mean', 'fracforest_median', 'fracirrigated_mean', 'fracirrigated_median', 'fracother_mean', 'fracother_median', 'ksat1_f_mean', 'ksat1_f_median', 'ksat1_o_mean', 'ksat1_o_median', 'tp_mean_annual', 'tp_std_interann', 'tp_seasonality', 'eT0_mean_annual', 'eT0_std_interann', 'eT0_seasonality', 'ta_mean', 'ta_std_interann', 'ta_seasonality', 'aridity_index', 'param_CalChanMan1', 'param_CalChanMan3', 'param_GwLoss', 'param_GwPercValue', 'param_LZThreshold', 'param_LakeMultiplier', 'param_LowerZoneTimeConsta

In [13]:
geometry = [Point(lon,lat) for lon,lat in zip(merged["lon"],merged["lat"])]
gdf = gpd.GeoDataFrame(merged, geometry=geometry, crs="EPSG:4326")
gdf.to_file("glofas5_hydrobot.geojson", index=False)
gdf.to_file(OUT_FOLD / "glofas5_hydrobot.geojson", index=False)
